In [1]:
!rm -r xCaliber

In [2]:
!git clone https://github.com/Pranshu-Bahadur/xCaliber.git

Cloning into 'xCaliber'...
remote: Enumerating objects: 740, done.
remote: Counting objects: 100% (269/269), done.
remote: Compressing objects: 100% (117/117), done.
remote: Total 740 (delta 118), reused 231 (delta 83), pack-reused 471 (from 1)
Receiving objects: 100% (740/740), 20.93 MiB | 31.80 MiB/s, done.
Resolving deltas: 100% (296/296), done.


In [3]:
%pip install ninja

In [4]:
import os

os.environ['TORCH_CUDA_ARCH_LIST'] = "7.5"

In [5]:
%cd xCaliber

/content/xCaliber


In [8]:
%cd ..

/content


In [6]:
!TORCH_SHOW_CPP_DETAILS=1 python xcaliber/setup.py build_ext --inplace

running build_ext
building 'xcalibur' extension
creating /content/xCaliber/build/temp.linux-x86_64-cpython-313/content/xCaliber/xcaliber/inference/fused-moe
[1/2] /usr/local/cuda/bin/nvcc -MD -MF /content/xCaliber/build/temp.linux-x86_64-cpython-313/content/xCaliber/xcaliber/inference/fused-moe/moe.o.d -I/usr/local/lib/python3.13/dist-packages/torch/include -I/usr/local/lib/python3.13/dist-packages/torch/include/torch/csrc/api/include -I/usr/local/cuda/include -I/usr/include/python3.13 -c -c /content/xCaliber/xcaliber/inference/fused-moe/moe.cu -o /content/xCaliber/build/temp.linux-x86_64-cpython-313/content/xCaliber/xcaliber/inference/fused-moe/moe.o -D__CUDA_NO_HALF_OPERATORS__ -D__CUDA_NO_HALF_CONVERSIONS__ -D__CUDA_NO_BFLOAT16_CONVERSIONS__ -D__CUDA_NO_HALF2_OPERATORS__ --expt-relaxed-constexpr --compiler-options ''"'"'-fPIC'"'"'' -O3 --use_fast_math -std=c++17 -DTORCH_API_INCLUDE_EXTENSION_H -DTORCH_EXTENSION_NAME=xcalibur -gencode=arch=compute_75,code=sm_75
[2/2] c++ -MMD -MF /co

In [9]:
import sys
!{sys.executable} -m pytest ./xCaliber/test/test_moe.py -q

F                                                                        [100%]
=================================== FAILURES ===================================
__________________________________ test_topk ___________________________________

    def test_topk():
        # The current binding dispatches FP32 input and launches on the default stream.
        with torch.cuda.stream(torch.cuda.default_stream()):
            for softmax in (False, True):
                for N in (8, 16, 16384):
                    for E in (128, 256, 384):
                        for K in (2, 8):
>                           check(N, E, K, softmax)

xCaliber/test/test_moe.py:72: 
_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ 

N = 8, E = 128, K = 2, softmax = False, kind = 'random', timed = False

    def check(N, E, K, softmax, kind="random", timed=False):
        torch.manual_seed(0)
        logits = torch.randn(N + 1, E, device="cuda", dtype=torch.float32) * 2
        if